In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

base_path = "/Users/omarsarayrah/Desktop/Comp333 project"

github_path = os.path.join(base_path, "github")
monthly_path = os.path.join(github_path, "monthly")
airline_file = os.path.join(github_path, "Airline.csv")
airport_file = os.path.join(github_path, "Airport.csv")


In [2]:
monthly_files = sorted([
    os.path.join(monthly_path, f)
    for f in os.listdir(monthly_path)
    if f.endswith(".csv")
])

df_list = []
for file in monthly_files:
    print("Loading:", file)
    df_month = pd.read_csv(file, low_memory=False)
    df_list.append(df_month)

flights = pd.concat(df_list, ignore_index=True)

print("Total flights:", flights.shape)


Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/01_january.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/02_february.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/03_march.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/04_april.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/05_may.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/06_june.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/07_july.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/08_august.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/09_september.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/10_october.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/11_november.csv
Loading: /Users/omarsarayrah/Desktop/Comp333 project/github/monthly/12_december.csv
Total flights:

In [3]:
airlines = pd.read_csv(airline_file)
airports = pd.read_csv(airport_file)

airlines.rename(columns={"IATA_CODE": "CARRIER_CODE", "AIRLINE": "CARRIER_NAME"}, inplace=True)
airports.rename(columns={"IATA_CODE": "AIRPORT_CODE"}, inplace=True)


In [4]:
flights.rename(columns={
    "OP_UNIQUE_CARRIER": "CARRIER_CODE",
    "ORIGIN": "ORIGIN",
    "DEST": "DEST"
}, inplace=True)


In [5]:
flights = flights.merge(airlines, on="CARRIER_CODE", how="left")


In [6]:
# ============================================
# CELL 6 — Select Columns + Create TOTAL_DELAY
# ============================================

# Columns we want if they exist
desired_cols = [
    "FL_DATE",
    "CARRIER_CODE", 
    "ORIGIN", "DEST",
    "CRS_DEP_TIME", "DEP_TIME",
    "CRS_ARR_TIME", "ARR_TIME",
    "DEP_DELAY", "ARR_DELAY",
    "CARRIER_DELAY", "WEATHER_DELAY",
    "NAS_DELAY", "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "CANCELLED"
]

# Keep only columns that actually exist in flights
available_cols = [c for c in desired_cols if c in flights.columns]

print("Columns kept in final dataset:")
print(available_cols)

# Extract reduced dataset
df_final = flights[available_cols].copy()

# ------------------------------------------------
# Create TOTAL_DELAY safely (handles missing cols)
# ------------------------------------------------
delay_cols = [
    "DEP_DELAY",
    "ARR_DELAY",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

# Keep only delay columns that exist
delay_cols_present = [c for c in delay_cols if c in df_final.columns]

print("Delay columns used to compute TOTAL_DELAY:")
print(delay_cols_present)

# Replace NaN with 0 for delay calculations
df_final[delay_cols_present] = df_final[delay_cols_present].fillna(0)

# Compute total delay
df_final["TOTAL_DELAY"] = df_final[delay_cols_present].sum(axis=1)

print("\nFinal dataset shape including TOTAL_DELAY:", df_final.shape)
df_final.head()


Columns kept in final dataset:
['FL_DATE', 'CARRIER_CODE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME', 'DEP_DELAY', 'ARR_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'CANCELLED']
Delay columns used to compute TOTAL_DELAY:
['DEP_DELAY', 'ARR_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

Final dataset shape including TOTAL_DELAY: (6847899, 17)


,FL_DATE,CARRIER_CODE,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,CRS_ARR_TIME,ARR_TIME,DEP_DELAY,ARR_DELAY,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,CANCELLED,TOTAL_DELAY
0,1/2/2023 12:00:00 AM,9E,BDL,LGA,800,757.0,905,853.0,-3.0,-12.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.0
1,1/2/2023 12:00:00 AM,9E,DLH,MSP,510,502.0,626,622.0,-8.0,-4.0,0.0,0.0,0.0,0.0,0.0,0.0,-12.0
2,1/2/2023 12:00:00 AM,9E,ORF,DTW,530,526.0,735,728.0,-4.0,-7.0,0.0,0.0,0.0,0.0,0.0,0.0,-11.0
3,1/2/2023 12:00:00 AM,9E,MSP,PIT,910,911.0,1220,1219.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1/2/2023 12:00:00 AM,9E,PIT,MSP,1310,1305.0,1440,1446.0,-5.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
output_path = "/Users/omarsarayrah/Desktop/Comp333 project/final_database_Comp333.csv"

df_final.to_csv(output_path, index=False)

print("Saved CSV to:")
print(output_path)


Saved CSV to:
/Users/omarsarayrah/Desktop/Comp333 project/final_database_Comp333.csv
